In [ ]:
"""
 [홍석환]
 공백·빈값, ID와 날짜 표기를 표준화하고 명시적 코드 매핑으로 서비스 분류를 정리하며 동일 내용 중복을 제거합니다. 
 가격 버전 ID, 유효기간 중첩과 단가 범위를 검증하고, 다른 버전의 명칭 또는 독립된 서비스 상세의 단가가 일치할 때만 누락 명칭과 오류 단가를 복구합니다. 
 변경 이력과 상세 단가 불일치 경고를 기록하여 정제본과 DQ 자료를 저장합니다.
 """

# service_catalog: 명세서·ERD 기반 정제
서비스 코드의 연도별 가격 버전입니다. 서비스 코드가 반복되는 것은 정상이며 기본키는 service_version_id입니다. 명세서 13_서비스 가격 버전, 18_참조 관계, 20_업무규칙·생성근거 B10·B15와 ERD를 적용합니다. 모든 코드는 이 노트북에 있습니다. .venv 커널로 실행하세요. 원본 수정·정제 결과 저장·DB 적재는 수행하지 않습니다.

## 1. 데이터 구조: (행, 열) 개수

In [1]:
from pathlib import Path
import csv
import polars as pl
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/input/service_catalog').is_dir())
INPUT=ROOT/'data/input'
COLS=['service_version_id','service_code','service_name','service_category','valid_from','valid_to','unit_price','source_system']
files=sorted((INPUT/'service_catalog').glob('*.csv'));assert files
for p in files:
    with p.open(encoding='utf-8-sig',newline='') as f:assert next(csv.reader(f))==COLS,p.name
raw=pl.concat([pl.scan_csv(p,infer_schema=False,row_index_name='source_record').with_columns(pl.lit(p.name).alias('source_file'),(pl.col('source_record')+1).alias('source_record')) for p in files]).collect()
print('Polars:',pl.__version__,'Files:',len(files),'Original shape:',(raw.height,len(COLS)))

Polars: 2.0.0 Files: 1 Original shape: (180, 8)


## 2. 데이터 내 컬럼 세부 정보
모든 컬럼은 NOT NULL이며 단가는 정수 KRW입니다. 실습용 합성 서비스로 실제 의료 수가나 진단·처방 코드가 아닙니다.

In [2]:
column_details=pl.DataFrame({'column':COLS,
    'description':['가격 버전 ID','연도별 반복 서비스 코드','합성 서비스명','서비스 분류','유효 시작일(포함)','유효 종료일(포함)','단가(KRW)','원천 시스템'],
    'mysql_dtype':['VARCHAR(32)','VARCHAR(32)','VARCHAR(100)','VARCHAR(16)','DATE','DATE','INT','VARCHAR(8)'],
    'constraint':['PK, SVV_9자리 숫자','SVC_9자리 숫자, 코드 단독 중복 제거 금지','필수, 최대 100자','CONSULT/TEST/PROCEDURE','유효한 날짜','시작일 이상','양수, INT 범위','LEGACY_A/LEGACY_B'],
    'null_count':[raw[c].null_count() for c in COLS],'unique_count':[raw[c].n_unique() for c in COLS]})
display(column_details)

shape: (8, 6)
┌──────────────────┬──────────────────┬──────────────┬─────────────────┬────────────┬──────────────┐
│ column           ┆ description      ┆ mysql_dtype  ┆ constraint      ┆ null_count ┆ unique_count │
│ ---              ┆ ---              ┆ ---          ┆ ---             ┆ ---        ┆ ---          │
│ str              ┆ str              ┆ str          ┆ str             ┆ i64        ┆ i64          │
╞══════════════════╪══════════════════╪══════════════╪═════════════════╪════════════╪══════════════╡
│ service_version_ ┆ 가격 버전 ID     ┆ VARCHAR(32)  ┆ PK, SVV_9자리   ┆ 0          ┆ 180          │
│ id               ┆                  ┆              ┆ 숫자            ┆            ┆              │
│ service_code     ┆ 연도별 반복      ┆ VARCHAR(32)  ┆ SVC_9자리 숫자, ┆ 0          ┆ 60           │
│                  ┆ 서비스 코드      ┆              ┆ 코드 단독 중복  ┆            ┆              │
│                  ┆                  ┆              ┆ 제거 금지       ┆            ┆              │
│ servic

## 3. 데이터 행(레코드) 샘플

In [3]:
display(raw.head(6))
display(raw.tail(5))
display(raw.sample(n=10,seed=42))

shape: (6, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source_re ┆ service_v ┆ service_c ┆ service_n ┆ … ┆ valid_to  ┆ unit_pric ┆ source_sy ┆ source_f │
│ cord      ┆ ersion_id ┆ ode       ┆ ame       ┆   ┆ ---       ┆ e         ┆ stem      ┆ ile      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ str       ┆ ---       ┆ ---       ┆ ---      │
│ u32       ┆ str       ┆ str       ┆ str       ┆   ┆           ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ SVV_00000 ┆ SVC_00000 ┆ 합성서비  ┆ … ┆ 2023-12-3 ┆ 1000      ┆ LEGACY_A  ┆ service_ │
│           ┆ 0001      ┆ 0001      ┆ 스_001    ┆   ┆ 1         ┆           ┆           ┆ catalog. │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ csv      │
│ 2         ┆ SVV_00000 ┆ SVC_00000 ┆ 합성서비  ┆ … ┆ 2024-12-3 ┆ 1200      ┆ LEGACY_

## 4. 클리닝 전 원본(오염된) 상태

In [4]:
raw_rules={}
for c in COLS:
    raw_rules['missing_'+c]=pl.col(c).is_null()|(pl.col(c).str.strip_chars()=='')
    raw_rules['whitespace_'+c]=pl.col(c)!=pl.col(c).str.strip_chars()
raw_rules['duplicate_content']=pl.struct(COLS).is_duplicated()
raw_rules['duplicate_version_id']=pl.col('service_version_id').is_duplicated()
for c in ['valid_from','valid_to']:
    raw_rules['invalid_date_'+c]=pl.col(c).is_not_null()&pl.col(c).str.to_date('%Y-%m-%d',strict=False).is_null()
raw_rules['invalid_price']=pl.col('unit_price').cast(pl.Int64,strict=False).is_null()|(pl.col('unit_price').cast(pl.Int64,strict=False)<=0)
raw_rules['nonstandard_category']=~pl.col('service_category').is_in(['CONSULT','TEST','PROCEDURE'])
flags=raw.select([e.fill_null(False).alias(k) for k,e in raw_rules.items()])
raw_report=pl.DataFrame({'rule':list(raw_rules),'rows':[flags[k].sum() for k in raw_rules]}).sort('rows',descending=True)
display(raw_report)
for k in ['missing_service_name','whitespace_service_name','nonstandard_category','invalid_price']:
    print(k);display(raw.filter(flags[k]))

shape: (22, 2)
┌───────────────────────────────┬──────┐
│ rule                          ┆ rows │
│ ---                           ┆ ---  │
│ str                           ┆ i64  │
╞═══════════════════════════════╪══════╡
│ missing_service_name          ┆ 4    │
│ invalid_price                 ┆ 4    │
│ missing_service_version_id    ┆ 0    │
│ whitespace_service_version_id ┆ 0    │
│ missing_service_code          ┆ 0    │
│ …                             ┆ …    │
│ duplicate_content             ┆ 0    │
│ duplicate_version_id          ┆ 0    │
│ invalid_date_valid_from       ┆ 0    │
│ invalid_date_valid_to         ┆ 0    │
│ nonstandard_category          ┆ 0    │
└───────────────────────────────┴──────┘
missing_service_name
shape: (4, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ source_re ┆ service_v ┆ service_c ┆ service_n ┆ … ┆ valid_to  ┆ unit_pric ┆ source_sy ┆ source_f │
│ cord      ┆ ersion_id ┆ ode       ┆ ame       ┆ 

## 5. 클리닝 로직/논리
1. 모든 원본 값과 행 위치를 보존하고 공백·빈 문자열, ID 하이픈, 슬래시 날짜를 표준화합니다.
2. 원천 시스템+도메인+원천 코드에 대한 명시적 매핑으로 서비스 분류를 표준화합니다. 미등록 코드나 충돌 매핑은 추측하지 않습니다.
3. 동일 내용 중복만 통합하고 가격 버전 ID 충돌을 검토합니다. 서비스 코드의 연도별 반복은 정상입니다.
4. 명칭 누락은 동일 서비스 코드의 다른 버전에서 동일 분류·동일 출처의 유효한 명칭이 최소 2개 버전에 걸쳐 하나로 일치할 때만 복구 후보로 사용합니다. 이는 동일 코드의 명칭 일관성을 적용한 추론이며 모든 변경을 기록합니다.
5. 동일 코드의 유효기간 중첩을 검증합니다. 기간 경계를 진료일 min/max나 연도 패턴으로 추측하지 않습니다.
6. 상세 내역은 중복·ID 충돌·수량·단가·줄 금액 검증 후 유일한 진료 시작일과 연결합니다. 실제 진료일이 카탈로그 유효기간 안에 있는 상세의 단가가 2건 이상에서 하나로 일치하면 단가 근거로 사용합니다. 상세의 기존 단가를 카탈로그 값으로 먼저 변경하지 않아 순환 근거를 방지합니다.
7. 기존 단가가 유효한 양수인데 상세와 다르면 단가를 임의로 덮어쓰지 않고 교차검증 경고로 남깁니다. 단가 파싱 실패·범위 오류에만 독립 상세 근거 복구를 허용합니다.
8. 참조 상세의 누락·오염 건수와 신뢰 가능한 상세의 단가·유효기간 불일치는 별도 보고합니다. 상세의 오류가 카탈로그 자체 오류라는 뜻은 아닙니다.

In [5]:
def strip_text(df,columns):
    return df.with_columns([pl.when(pl.col(c).str.strip_chars()=='').then(None).otherwise(pl.col(c).str.strip_chars()).alias(c) for c in columns])
def canonical_ids(df,columns):
    return df.with_columns([pl.col(c).str.replace_all('-','_').alias(c) for c in columns])
def load(table,ids):
    f=sorted((INPUT/table).glob('*.csv'));assert f,table
    d=pl.scan_csv(f,infer_schema=False).collect()
    return canonical_ids(strip_text(d,d.columns),ids).unique(maintain_order=True)
def unique_reference(df,key):
    return df.filter(pl.col(key).is_not_null()&(pl.len().over(key)==1))
text=canonical_ids(strip_text(raw.with_columns([pl.col(c).alias('raw_'+c) for c in COLS]),COLS),['service_version_id','service_code']).with_columns([
    pl.col(c).str.replace_all('/','-').alias(c) for c in ['valid_from','valid_to']])
mappings=load('legacy_code_mappings',['mapping_id'])
m=mappings.filter(pl.col('domain')=='service_category').select('source_system',pl.col('raw_code').alias('service_category'),'standard_code').unique()
m=m.filter(pl.len().over(['source_system','service_category'])==1)
text=text.join(m,on=['source_system','service_category'],how='left',validate='m:1').with_columns(pl.coalesce('standard_code','service_category').alias('service_category')).drop('standard_code')
duplicate_rows=text.filter(pl.struct(COLS).is_duplicated())
base=text.unique(subset=COLS,maintain_order=True).with_columns(pl.col('valid_from').str.to_date('%Y-%m-%d',strict=False),pl.col('valid_to').str.to_date('%Y-%m-%d',strict=False),pl.col('unit_price').cast(pl.Int64,strict=False))
removed=raw.height-base.height
# 동일 코드의 다른 가격 버전에서 이름 일관성 확인
names=base.filter(pl.col('service_name').is_not_null()&(pl.col('service_name').str.len_chars()<=100)
    &pl.col('service_code').str.contains(r'^SVC_\d{9}$')&pl.col('service_version_id').str.contains(r'^SVV_\d{9}$')
    &(pl.len().over('service_version_id')==1)&pl.col('service_category').is_in(['CONSULT','TEST','PROCEDURE'])
    &(pl.col('valid_from')<=pl.col('valid_to'))&pl.col('unit_price').is_between(1,2147483647))
name_evidence=names.group_by('service_code','service_category','source_system').agg(
    pl.col('service_name').n_unique().alias('name_variants'),pl.col('service_version_id').n_unique().alias('name_version_count'),pl.col('service_name').first().alias('candidate_name'))
work=base.join(name_evidence,on=['service_code','service_category','source_system'],how='left',validate='m:1').with_columns(pl.col('service_name').alias('before_service_name'),pl.col('unit_price').alias('before_unit_price'))
work=work.with_columns(pl.when(pl.col('service_name').is_null()&(pl.col('name_variants')==1)&(pl.col('name_version_count')>=2)).then(pl.col('candidate_name')).otherwise(pl.col('service_name')).alias('service_name'))
# 동일 코드의 기간 중첩을 모든 조합으로 검사
periods=base.filter(pl.col('service_code').is_not_null()&pl.col('valid_from').is_not_null()&pl.col('valid_to').is_not_null())
overlap_pairs=periods.select('service_code','service_version_id','valid_from','valid_to').join(periods.select('service_code',pl.col('service_version_id').alias('other_version'),pl.col('valid_from').alias('other_from'),pl.col('valid_to').alias('other_to')),on='service_code').filter(
    (pl.col('service_version_id')!=pl.col('other_version'))&(pl.col('valid_from')<=pl.col('other_to'))&(pl.col('other_from')<=pl.col('valid_to')))
overlap_ids=overlap_pairs.select('service_version_id').unique().with_columns(pl.lit(True).alias('period_overlap'))
work=work.join(overlap_ids,on='service_version_id',how='left',validate='m:1')

In [6]:
services=load('encounter_services',['encounter_service_id','encounter_id','service_version_id']).with_columns([
    pl.col(c).cast(pl.Int64,strict=False) for c in ['line_no','quantity','unit_price','line_amount']])
enc=load('encounters',['encounter_id','checkin_id']).with_columns(pl.col('started_at').str.replace_all('/','-').str.to_datetime('%Y-%m-%d %H:%M:%S',strict=False).dt.date().alias('encounter_date'))
enc=unique_reference(enc,'encounter_id').select('encounter_id','encounter_date')
services=services.join(enc,on='encounter_id',how='left',validate='m:1').with_columns(
    (pl.col('encounter_service_id').str.contains(r'^ESV_\d{9}$')&(pl.len().over('encounter_service_id')==1)
     &pl.col('encounter_id').is_not_null()&(pl.len().over(['encounter_id','line_no'])==1)
     &pl.col('line_no').is_between(1,2147483647)&pl.col('quantity').is_between(1,2147483647)
     &pl.col('unit_price').is_between(1,2147483647)&pl.col('line_amount').is_between(1,2147483647)
     &(pl.col('line_amount')==pl.col('quantity')*pl.col('unit_price'))&pl.col('encounter_date').is_not_null()
     &pl.col('source_system').is_in(['LEGACY_A','LEGACY_B'])).fill_null(False).alias('detail_valid'))
cat_ref=unique_reference(base,'service_version_id').select('service_version_id','valid_from','valid_to',pl.col('unit_price').alias('catalog_price'))
services=services.join(cat_ref,on='service_version_id',how='left',validate='m:1').with_columns(
    pl.col('encounter_date').is_between(pl.col('valid_from'),pl.col('valid_to'),closed='both').fill_null(False).alias('date_valid'))
price_evidence=services.filter(pl.col('detail_valid')&pl.col('date_valid')).group_by('service_version_id').agg(
    pl.col('unit_price').n_unique().alias('price_variants'),pl.len().alias('price_support_rows'),pl.col('unit_price').first().alias('candidate_price'))
work=work.join(price_evidence,on='service_version_id',how='left',validate='m:1').with_columns(
    pl.when((pl.col('unit_price').is_null()|(~pl.col('unit_price').is_between(1,2147483647)))
    &(pl.col('price_variants')==1)&(pl.col('price_support_rows')>=2)&(~pl.col('period_overlap').fill_null(False)))
    .then(pl.col('candidate_price')).otherwise(pl.col('unit_price')).alias('unit_price'))
reference_report=services.select(
    pl.len().alias('detail_rows'),(~pl.col('detail_valid')).sum().alias('invalid_detail_rows'),
    (pl.col('valid_from').is_null()|pl.col('valid_to').is_null()).sum().alias('catalog_reference_unavailable'),
    (pl.col('detail_valid')&pl.col('valid_from').is_not_null()&(~pl.col('date_valid'))).sum().alias('outside_validity'),
    (pl.col('detail_valid')&pl.col('date_valid')&(pl.col('unit_price')!=pl.col('catalog_price'))).sum().alias('price_mismatch'))
# 복구 후 카탈로그 단가로 동일 상세 근거를 재검증
final_prices=unique_reference(work,'service_version_id').select('service_version_id',pl.col('unit_price').alias('final_catalog_price'))
services_final=services.join(final_prices,on='service_version_id',how='left',validate='m:1')
reference_report=reference_report.rename({'price_mismatch':'price_mismatch_before_repair'}).with_columns(
    pl.lit(services_final.filter(pl.col('detail_valid')&pl.col('date_valid')&(pl.col('unit_price')!=pl.col('final_catalog_price'))).height).alias('price_mismatch_after_repair'))

In [7]:
audit=[]
for c in ['service_name','unit_price']:
    audit.append(work.filter(~pl.col(c).eq_missing(pl.col('before_'+c))).select('service_version_id','source_file','source_record',pl.lit(c).alias('column'),pl.col('raw_'+c).alias('original_value'),pl.col(c).cast(pl.String).alias('clean_value'),pl.lit('same-code versions unanimous' if c=='service_name' else 'independent valid detail prices unanimous').alias('evidence')))
for c in COLS:
    audit.append(text.filter(~pl.col(c).eq_missing(pl.col('raw_'+c))).select('service_version_id','source_file','source_record',pl.lit(c).alias('column'),pl.col('raw_'+c).alias('original_value'),pl.col(c).alias('clean_value'),pl.lit('B15 notation or explicit code mapping').alias('evidence')))
repair_log=pl.concat(audit,how='vertical_relaxed')
rules={}
for c in COLS:rules['missing_or_unparseable_'+c]=pl.col(c).is_null()
rules.update({
    'invalid_version_id':~pl.col('service_version_id').str.contains(r'^SVV_\d{9}$'),
    'invalid_service_code':~pl.col('service_code').str.contains(r'^SVC_\d{9}$'),
    'duplicate_version_id':pl.col('service_version_id').is_not_null()&(pl.len().over('service_version_id')>1),
    'invalid_name_length':pl.col('service_name').str.len_chars()>100,
    'invalid_category':~pl.col('service_category').is_in(['CONSULT','TEST','PROCEDURE']),
    'invalid_source_system':~pl.col('source_system').is_in(['LEGACY_A','LEGACY_B']),
    'invalid_period':pl.col('valid_from')>pl.col('valid_to'),
    'period_overlap':pl.col('period_overlap').fill_null(False),
    'invalid_price':~pl.col('unit_price').is_between(1,2147483647),
})
checked=work.with_columns([e.fill_null(False).alias(k) for k,e in rules.items()]).with_columns(
    pl.concat_list([pl.when(pl.col(k)).then(pl.lit(k)).otherwise(None) for k in rules]).list.drop_nulls().alias('issue_reasons'),
    ((pl.col('price_variants')>1)|((pl.col('price_variants')==1)&(pl.col('candidate_price')!=pl.col('unit_price')))).fill_null(False).alias('detail_price_disagreement_warning'))
quality_report=pl.DataFrame({'rule':list(rules),'rows':[checked[k].sum() for k in rules]}).sort('rows',descending=True)

## 6. 클리닝 결과
구조·업무 규칙 통과와 상세 참조 경고를 구분합니다. 카탈로그 단가가 유효해도 상세 단가가 다르면 어느 쪽이 오염인지 현재 근거로 확정할 수 없으므로 경고를 남깁니다. 관련 상세를 카탈로그에 맞춰 자동 덮어쓰지 않습니다.

In [8]:
service_catalog_clean=checked.filter(pl.col('issue_reasons').list.len()==0).select(COLS+['source_file','source_record','detail_price_disagreement_warning'])
service_catalog_review=checked.filter(pl.col('issue_reasons').list.len()>0)
assert raw.height==removed+service_catalog_clean.height+service_catalog_review.height
assert service_catalog_clean['service_version_id'].n_unique()==service_catalog_clean.height
assert service_catalog_clean.select(pl.col(COLS).null_count()).row(0)==(0,)*len(COLS)
summary=pl.DataFrame({'stage':['raw','duplicate_removed','clean','review'],'rows':[raw.height,removed,service_catalog_clean.height,service_catalog_review.height]})
repair_summary=repair_log.group_by('column','evidence').len().sort('column')
display(summary);display(quality_report);display(repair_summary);display(repair_log)
display(reference_report);display(overlap_pairs)
display(service_catalog_clean.head(10));display(service_catalog_review)
print('가격 혼재 경고 버전 수:',checked['detail_price_disagreement_warning'].sum())

shape: (4, 2)
┌───────────────────┬──────┐
│ stage             ┆ rows │
│ ---               ┆ ---  │
│ str               ┆ i64  │
╞═══════════════════╪══════╡
│ raw               ┆ 180  │
│ duplicate_removed ┆ 0    │
│ clean             ┆ 180  │
│ review            ┆ 0    │
└───────────────────┴──────┘
shape: (17, 2)
┌─────────────────────────────────┬──────┐
│ rule                            ┆ rows │
│ ---                             ┆ ---  │
│ str                             ┆ i64  │
╞═════════════════════════════════╪══════╡
│ missing_or_unparseable_service… ┆ 0    │
│ missing_or_unparseable_service… ┆ 0    │
│ missing_or_unparseable_service… ┆ 0    │
│ missing_or_unparseable_service… ┆ 0    │
│ missing_or_unparseable_valid_f… ┆ 0    │
│ …                               ┆ …    │
│ invalid_category                ┆ 0    │
│ invalid_source_system           ┆ 0    │
│ invalid_period                  ┆ 0    │
│ period_overlap                  ┆ 0    │
│ invalid_price                   ┆ 

## 7. 정제 결과 파일 저장
최종 정제 데이터는 data/output/service_catalog.csv에 업무 컬럼만 저장합니다. Polars는 pandas의 인덱스와 같은 별도 번호 컬럼을 생성하지 않습니다. 추적용 source_record도 최종 파일에서 제외합니다.

UTF-8 BOM(utf-8-sig), 날짜 YYYY-MM-DD, 일시 YYYY-MM-DD HH:MM:SS를 사용하고 실제 결측값은 빈칸으로 저장합니다. 검토 대상·변경 이력·검증 보고서는 data/output/reports/service_catalog/에 같은 기준으로 저장합니다. 원본 값을 기록한 검토 자료에는 오염된 문자열이 그대로 남을 수 있습니다.

각 노트북은 기존 정제 규칙으로 독립 실행하며, 다른 노트북의 복구 결과가 자동 연결되는 것은 아닙니다. MySQL 적재는 수행하지 않습니다.

최종 정제본에서 제외된 검토 대상과 실제 중복 제거 행은 data/output/service_catalog_dq.csv에 합쳐 저장합니다. dq_type은 REVIEW 또는 DUPLICATE_EXCLUDED, dq_reason은 검토 사유입니다. 업무 컬럼에는 수정 전 원본 값을 넣고 파일명과 레코드 순번을 남깁니다. 중복 그룹의 대표 행은 중복 제외 목록에 넣지 않습니다. 제외 행이 없더라도 헤더가 있는 빈 파일을 생성합니다.

In [9]:
from pathlib import Path
import json
FINAL_DIR = ROOT / 'data/output'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = FINAL_DIR / 'reports' / 'service_catalog'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def export_csv(frame, filename):
    # CSV에 중첩 자료형을 직접 저장할 수 없어 사유 목록을 구분 문자열로 보존
    out = frame.with_columns([pl.col(c).list.join('|').alias(c)
        for c, dtype in frame.schema.items() if isinstance(dtype, pl.List)])
    destination = FINAL_DIR / filename if filename in ['service_catalog.csv', 'service_catalog_dq.csv'] else OUTPUT_DIR / filename
    out.write_csv(destination, include_bom=True, null_value='',
        datetime_format='%Y-%m-%d %H:%M:%S', date_format='%Y-%m-%d')
    return {'file': filename, 'rows': frame.height, 'columns': frame.width}
exported = []
business_columns = COLS
exported.append(export_csv(service_catalog_clean.select(business_columns), 'service_catalog.csv'))
# DQ에는 검토 행과 실제 제거된 중복 행을 원본 값으로 기록
provenance = ['source_file', 'source_record']
retained_locations = base.select(provenance)
excluded_locations = text.select(provenance).join(retained_locations, on=provenance, how='anti')
review_reasons = service_catalog_review.select(provenance + ['issue_reasons']).with_columns(
    pl.col('issue_reasons').list.join('|').alias('dq_reason')).drop('issue_reasons')
dq_review = raw.join(review_reasons, on=provenance, how='inner', validate='1:1').with_columns(
    pl.lit('REVIEW').alias('dq_type'))
dq_duplicates = raw.join(excluded_locations, on=provenance, how='inner', validate='1:1').with_columns(
    pl.lit('DUPLICATE_EXCLUDED').alias('dq_type'),
    pl.lit('identical_normalized_content').alias('dq_reason'))
dq_columns = business_columns + provenance + ['dq_type', 'dq_reason']
service_catalog_dq = pl.concat([dq_review.select(dq_columns), dq_duplicates.select(dq_columns)])
expected_excluded = summary.filter(pl.col('stage') == 'duplicate_removed')['rows'][0]
assert dq_duplicates.height == expected_excluded
assert dq_review.height == service_catalog_review.height
assert raw.height == service_catalog_clean.height + service_catalog_dq.height
assert service_catalog_dq.select(provenance).unique().height == service_catalog_dq.height
assert service_catalog_dq.select(provenance).join(service_catalog_clean.select(provenance), on=provenance, how='inner').is_empty()
exported.append(export_csv(service_catalog_dq, 'service_catalog_dq.csv'))
loaded_dq = pl.read_csv(FINAL_DIR / 'service_catalog_dq.csv', infer_schema=False)
assert loaded_dq.height == service_catalog_dq.height and loaded_dq.columns == dq_columns

exported.append(export_csv(service_catalog_review, 'review.csv'))
for variable in ['summary', 'quality_report', 'raw_report', 'repair_log', 'repair_summary',
                 'reconciliation', 'reconciliation_report', 'reference_report',
                 'duplicate_rows', 'seen_without_encounter']:
    value = globals().get(variable)
    if isinstance(value, pl.DataFrame):
        exported.append(export_csv(value, variable + '.csv'))
# 전송 결과의 행 수 및 스키마 확인 (자료형은 CSV 밖 manifest에 기록)
loaded = pl.read_csv(FINAL_DIR / 'service_catalog.csv', infer_schema=False)
assert loaded.height == service_catalog_clean.height and loaded.columns == business_columns
manifest = {'table': 'service_catalog', 'polars_version': pl.__version__,
    'clean_schema': {c: str(dtype) for c, dtype in service_catalog_clean.select(business_columns).schema.items()},
    'null_token': '', 'list_separator': '|', 'encoding': 'utf-8-sig',
    'execution_mode': 'independent notebooks using original reference data', 'files': exported}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Final CSV:', FINAL_DIR / 'service_catalog.csv')
print('Reports:', OUTPUT_DIR)
display(pl.DataFrame(exported))


Final CSV: C:\workspace\medallion_venv\data\output\service_catalog.csv
Reports: C:\workspace\medallion_venv\data\output\reports\service_catalog
shape: (10, 3)
┌────────────────────────┬──────┬─────────┐
│ file                   ┆ rows ┆ columns │
│ ---                    ┆ ---  ┆ ---     │
│ str                    ┆ i64  ┆ i64     │
╞════════════════════════╪══════╪═════════╡
│ service_catalog.csv    ┆ 180  ┆ 8       │
│ service_catalog_dq.csv ┆ 0    ┆ 12      │
│ review.csv             ┆ 0    ┆ 45      │
│ summary.csv            ┆ 4    ┆ 2       │
│ quality_report.csv     ┆ 17   ┆ 2       │
│ raw_report.csv         ┆ 22   ┆ 2       │
│ repair_log.csv         ┆ 11   ┆ 7       │
│ repair_summary.csv     ┆ 3    ┆ 3       │
│ reference_report.csv   ┆ 1    ┆ 6       │
│ duplicate_rows.csv     ┆ 0    ┆ 18      │
└────────────────────────┴──────┴─────────┘
